# Monthly Landsat AdDSWE Generator (30 m)

Generates the monthly Adapted Dynamic Surface Water Extent (AdDSWE) inundation record
for the Okavango Delta from Landsat Collection 2 surface reflectance, and exports each
product to Earth Engine assets.

The core algorithm lives in the `addswe` package (`classification.py`, `composites.py`);
this notebook imports it and provides the export orchestration and run configuration.

**Products** (per month, each to its own asset subfolder): AdDSWE class raster (`DSWE`),
QC/temporal-expansion mask (`QC`), RGB source composite (`Composite`), and optionally the
original 5-test DSWE (`DSWE` under `Original_5Test_DSWE_Products`, for the Fig. 7 comparison).

> **Not turnkey.** Requires a Google Earth Engine account/project, authentication, and
> long-running export tasks. Set the asset paths and study-area shapefile in the config
> cell before running.


## Imports

In [ ]:
import ee
import calendar
import logging
from datetime import datetime, timedelta

from addswe import (
    getLandsatCollection,
    maskL8sr,
    rescale,
    load_roi,
    get_filled_composite_before_dswe,
    Dswe_with_Test6,
    morphological_filter,
)

ee.Initialize()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


## Configuration

In [ ]:
# =========================== CONFIGURATION (edit before running) ===========================

# ---- Study period ----
start_date = datetime(1984, 6, 1)
end_date   = datetime(2025, 12, 31)

# ---- Paths / GEE assets ----
study_area_path = r"Delta_UCB_WGS84.shp"   # delta boundary shapefile (Dryad: study_areas/)
gee_asset_output_folder = "projects/ee-okavango/assets/water_masks/monthly_DSWE_Landsat_30m_v4"

# ---- Products to export (each to its own subfolder of the asset folder) ----
export_dswe          = True    # AdDSWE (Test 6 + morphological filter) -- primary product
export_qc            = True    # QC / temporal-expansion mask
export_composite     = True    # RGB source composite
export_original_dswe = True    # original 5-test DSWE (no Test 6) -- used for the Fig. 7 comparison
export_swir2         = False
export_unfiltered_dswe = False

# ---- Test 6 (dynamic SWIR2 threshold) ----
min_swir2 = 0.04
max_swir2 = 0.15
save_swir2_plots = False
plot_output_dir  = None

# ---- Morphological filter ----
apply_morphological_filter = True
blob_size_threshold      = 2000   # pixels (~180 ha at 30 m); value used in the manuscript
blob_max_class_threshold = 2      # preserve any blob containing a pixel of class > 2


## Water-masking orchestration functions

In [ ]:
def get_water_mask_for_feature(start_date, end_date, polygon, 
                                min_swir2=0.04, max_swir2=0.15,
                                save_plot=True, output_dir=None, 
                                year=None, month=None,
                                return_original=False):
    """
    Generate a water mask for a given date range and polygon feature using Landsat imagery
    with DSWE Test 6 enhancement for vegetated inundation.
    
    Parameters:
    -----------
    start_date : str
        Start date in 'YYYY-MM-DD' format
    end_date : str
        End date in 'YYYY-MM-DD' format
    polygon : ee.Geometry.Polygon
        The polygon feature defining the area of interest
    min_swir2, max_swir2 : float, optional
        Safety constraints on SWIR2 threshold (default: 0.04 to 0.15)
    save_plot : bool, optional (default=True)
        Whether to save SWIR2 histogram plot
    output_dir : str, optional (default=None)
        Directory to save plots
    year : int, optional
        Year for metadata and plot filename
    month : int, optional
        Month for metadata and plot filename
    return_original : bool, optional (default=False)
        If True, returns tuple of (upgraded_mask, original_mask, threshold)
        If False, returns only upgraded_mask
    
    Returns:
    --------
    ee.Image or tuple : 
        If return_original=False: upgraded water mask image
        If return_original=True: (upgraded_mask, original_mask, swir2_threshold)
    """
    
    # Build Landsat composite
    imagery = (getLandsatCollection()
               .map(maskL8sr)
               .map(rescale)
               .filterDate(start_date, end_date)
               .filterBounds(polygon))
    
    image_composite = imagery.median().clip(polygon)
    
    # Apply DSWE with Test 6
    upgraded_mask, original_mask, swir2_threshold = Dswe_with_Test6(
        image_composite,
        polygon,
        min_swir2=min_swir2,
        max_swir2=max_swir2,
        save_plot=save_plot,
        output_dir=output_dir,
        year=year,
        month=month
    )
    
    if return_original:
        return upgraded_mask, original_mask, swir2_threshold
    else:
        return upgraded_mask


def export_to_asset(image, year, month, roi, asset_folder, bands=None, product_name="DSWE"):
    """
    Export a single-band or multi-band Earth Engine image to a GEE asset, 
    with proper metadata and asset ID formatting.
    
    Parameters:
        image (ee.Image): The image to export
        year (int): The image year
        month (int): The image month
        roi (ee.Geometry): Region of interest for clipping/export
        asset_folder (str): GEE asset folder path (no trailing slash)
        bands (list of str): Specific bands to export
        product_name (str): Prefix for the image asset name (e.g. "DSWE", "QC", "Composite")
    """
    asset_id = f"{asset_folder}/{product_name}_{year}_{month:02d}"
    
    # Select only specified bands if provided
    if bands:
        image_to_export = image.select(bands)
    else:
        image_to_export = image

    # Compute last day of month
    last_day = calendar.monthrange(year, month)[1]

    # Prepare image metadata
    image_with_metadata = image_to_export.set({
        'system:time_start': ee.Date(f"{year}-{month:02d}-01").millis(),
        'system:time_end': ee.Date(f"{year}-{month:02d}-{last_day:02d}").millis(),
        'source_ids': image.get('source_ids')
    })

    # Check if the asset already exists
    try:
        ee.data.getAsset(asset_id)
        logging.info(f"Skipping {asset_id}, already exists.")
    except:
        # Export the image with metadata
        task = ee.batch.Export.image.toAsset(
            image=image_with_metadata,
            description=f"{product_name}_{year}_{month:02d}",
            assetId=asset_id,
            scale=30,
            region=roi,
            maxPixels=1e13
        )
        task.start()
        logging.info(f"Exporting {asset_id}...")


def process_monthly_dswe(start_date, end_date, shapefile_path, asset_folder,
                         export_dswe=True, export_original_dswe=False, 
                         export_qc=True, export_composite=True, export_swir2=True,
                         min_swir2=0.04, max_swir2=0.15,
                         save_swir2_plots=True, plot_output_dir=None,
                         apply_morphological_filter=True,
                         blob_size_threshold=50,
                         blob_max_class_threshold=2,
                         export_unfiltered_dswe=False):
    
    """
    Generate and export DSWE composites with Test 6 enhancement for each month within 
    the given date range, with optional export of DSWE, original DSWE, QC, RGB composite, 
    and SWIR2 products, each to its own fixed subfolder.
    
    Parameters:
    -----------
    start_date : datetime
        Start date for processing
    end_date : datetime
        End date for processing
    shapefile_path : str
        Path to the ROI shapefile
    asset_folder : str
        Base folder path for asset exports
    export_dswe : bool, optional (default=True)
        Whether to export DSWE water classification products (with Test 6 enhancement)
    export_original_dswe : bool, optional (default=False)
        Whether to export original DSWE classification (without Test 6) for comparison
    export_qc : bool, optional (default=True)
        Whether to export QC mask products
    export_composite : bool, optional (default=True)
        Whether to export RGB Landsat composite products
    export_swir2 : bool, optional (default=True)
        Whether to export SWIR2 band products
    min_swir2, max_swir2 : float, optional
        Safety constraints on SWIR2 threshold (default: 0.04 to 0.15)
    save_swir2_plots : bool, optional (default=True)
        Whether to save SWIR2 histogram plots with threshold
    plot_output_dir : str, optional (default=None)
        Directory to save SWIR2 plots. If None, saves to current directory
    apply_morphological_filter : bool, optional (default=True)
        Whether to apply morphological filtering to remove isolated low-confidence blobs
    blob_size_threshold : int, optional (default=50)
        Maximum blob size (in pixels) for removal consideration in morphological filter
        At 30m resolution: 50 pixels ≈ 4.5 hectares
    blob_max_class_threshold : int, optional (default=2)
        Maximum DSWE class - blobs with values > this are preserved regardless of size
    export_unfiltered_dswe : bool, optional (default=False)
        Whether to export DSWE before morphological filtering (for comparison/validation)
    """
    # Fixed subfolder names
    # Fixed subfolder names
    DSWE_FOLDER = "DSWE_Products"
    DSWE_ORIGINAL_FOLDER = "Original_5Test_DSWE_Products"
    DSWE_UNFILTERED_FOLDER = "DSWE_Unfiltered_Products"
    COMPOSITE_FOLDER = "Source_LS_Composites"
    QC_FOLDER = "QC_Masks"
    SWIR2_FOLDER = "SWIR_2_LS"
    
    roi = load_roi(shapefile_path)
    current_date = start_date
    
    while current_date <= end_date:
        year, month = current_date.year, current_date.month
        last_day = calendar.monthrange(year, month)[1]
        im_start_date = f"{year}-{month:02d}-01"
        im_end_date = f"{year}-{month:02d}-{last_day:02d}"
        
        try:
            # Get filled composite
            filled_composite, expansion_mask = get_filled_composite_before_dswe(
                im_start_date, im_end_date, roi
            )
            
            # Apply DSWE with Test 6
            logging.info(f"Processing DSWE with Test 6 for {year}-{month:02d}...")
            dswe_upgraded, dswe_original, swir2_threshold = Dswe_with_Test6(
                filled_composite,
                roi,
                min_swir2=min_swir2,
                max_swir2=max_swir2,
                save_plot=save_swir2_plots,
                output_dir=plot_output_dir,
                year=year,
                month=month
            )
            
            logging.info(f"  SWIR2 threshold: {swir2_threshold:.4f}")
            
            # Apply morphological filter if enabled
            if apply_morphological_filter:
                logging.info(f"  Applying morphological filter...")
                dswe_filtered, filter_diagnostics = morphological_filter(
                    dswe_upgraded,
                    size_threshold=blob_size_threshold,
                    max_class_threshold=blob_max_class_threshold,
                    roi=roi,
                    return_diagnostics=True
                )
                
                # Log diagnostics
                if filter_diagnostics:
                    logging.info(f"    Removed {filter_diagnostics['pixels_removed']} pixels "
                               f"({filter_diagnostics['area_removed_km2']} km²)")
                    logging.info(f"    Class breakdown: {filter_diagnostics['class_1_pixels_removed']} "
                               f"class 1, {filter_diagnostics['class_2_pixels_removed']} class 2")
                    
                    # WARNING if high-confidence pixels were removed (should never happen)
                    if filter_diagnostics['class_3_pixels_removed'] > 0 or filter_diagnostics['class_4_pixels_removed'] > 0:
                        logging.warning(f"    WARNING: High-confidence pixels removed! "
                                      f"Class 3: {filter_diagnostics['class_3_pixels_removed']}, "
                                      f"Class 4: {filter_diagnostics['class_4_pixels_removed']}")
                
                # Store unfiltered version if needed for export
                dswe_unfiltered = dswe_upgraded
                dswe_upgraded = dswe_filtered
            else:
                logging.info(f"  Morphological filter disabled - skipping")
                dswe_unfiltered = None
            
            # Check for non-empty result
            if hasattr(dswe_upgraded, 'size') and dswe_upgraded.size().getInfo() == 0:
                logging.warning(f"No data available for {year}-{month:02d}.")
            else:
                # Export upgraded DSWE (with Test 6 + morphological filter if applied)
                if export_dswe:
                    export_to_asset(
                        dswe_upgraded,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{DSWE_FOLDER}",
                        bands=["dswe"],
                        product_name="DSWE"
                    )
                
                # Export unfiltered DSWE (with Test 6, before morphological filter)
                if export_unfiltered_dswe and dswe_unfiltered is not None:
                    export_to_asset(
                        dswe_unfiltered,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{DSWE_UNFILTERED_FOLDER}",
                        bands=["dswe"],
                        product_name="DSWE_Unfiltered"
                    )
                
                # Export original DSWE (without Test 6) for comparison
                if export_original_dswe:
                    export_to_asset(
                        dswe_original,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{DSWE_ORIGINAL_FOLDER}",
                        bands=["dswe"],
                        product_name="DSWE"
                    )
                
                # Export QC raster
                if export_qc:
                    export_to_asset(
                        expansion_mask,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{QC_FOLDER}",
                        bands=["expansion_mask"],
                        product_name="QC"
                    )
                
                # Export RGB Landsat composite
                if export_composite:
                    export_to_asset(
                        filled_composite,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{COMPOSITE_FOLDER}",
                        bands=["Blue", "Green", "Red"],
                        product_name="Composite"
                    )
                
                # Export SWIR2 band
                if export_swir2:
                    export_to_asset(
                        filled_composite,
                        year,
                        month,
                        roi,
                        f"{asset_folder}/{SWIR2_FOLDER}",
                        bands=["Swir2"],
                        product_name="SWIR2"
                    )
                    
        except Exception as e:
            logging.warning(f"Failed to process {year}-{month:02d}: {e}")
        
        # Move to first of the next month
        current_date = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1)

## Run: generate and export monthly products

In [ ]:
process_monthly_dswe(
    start_date,
    end_date,
    study_area_path,
    gee_asset_output_folder,
    export_dswe=export_dswe,
    export_original_dswe=export_original_dswe,
    export_qc=export_qc,
    export_composite=export_composite,
    export_swir2=export_swir2,
    min_swir2=min_swir2,
    max_swir2=max_swir2,
    save_swir2_plots=save_swir2_plots,
    plot_output_dir=plot_output_dir,
    apply_morphological_filter=apply_morphological_filter,
    blob_size_threshold=blob_size_threshold,
    blob_max_class_threshold=blob_max_class_threshold,
    export_unfiltered_dswe=export_unfiltered_dswe,
)


## Monitor Earth Engine tasks

In [ ]:
task_list = ee.batch.Task.list()

for task in task_list:
    print(f"Task: {task.status()['description']}, Status: {task.status()['state']}")
